# Insurance Fraud Detection — Experiment Notebook

> **Paper:** *Safe Learning to Defer with CVaR-Penalised Bayesian Uncertainty*

This notebook reproduces all experimental results and figures. It is organised in two parts:

- **Part A** — Run the full pipeline end-to-end (one cell).
- **Part B** — Interactive figure generation (one cell per figure, 9 total).

---
# Part A — Run Full Pipeline

Runs `fraud_model_advanced.main()`, which trains 6 classifiers, performs Bayesian/causal analysis, and writes:
- `reports/fraud_model_advanced_report.md`
- 9 SVG figures → `figures/`

In [1]:
%matplotlib inline
import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import fraud_model_advanced
fraud_model_advanced.main()

ModuleNotFoundError: No module named 'xgboost'

---
# Part B — Interactive Figures

Run the **Setup** cell once, then run any individual figure cell to inspect or tweak it.
All figures save to `figures/`.

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'svg'

import os, sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import networkx as nx
from scipy.stats import gaussian_kde

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score, roc_curve, auc
from sklearn.calibration import calibration_curve

from fraud_model import (
    load_and_clean, engineer_features, classify_columns,
    build_preprocessor, RANDOM_STATE, TEST_SIZE,
)
from fraud_model_advanced import (
    compute_class_conditionals, compute_posterior_updates,
    get_all_models, run_mcnemar_matrix, run_cross_validation,
    build_causal_dag, propensity_score_analysis, compute_feature_importance_comparison,
    BAYESIAN_FEATURES, TREATMENT_VAR, CONFOUNDERS, CSV_PATH, COLORS as ADV_COLORS,
)

# ── Save helper ──
SAVE_DIR = os.path.join(os.path.dirname(os.path.abspath("__file__")), "figures")
os.makedirs(SAVE_DIR, exist_ok=True)

def save_fig(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, f"{name}.svg"), format="svg", bbox_inches="tight")
    print(f"  Saved: figures/{name}.svg")

# ── 1. Load & preprocess ──
print("[1/5] Loading data...")
df = load_and_clean(CSV_PATH)
df = engineer_features(df)
df_full = df.copy()
X, y, num_cols, low_cat_cols, med_cat_cols = classify_columns(df, "fraud_reported")
preprocessor = build_preprocessor(num_cols, low_cat_cols, med_cat_cols)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y)
prior_fraud = y_train.mean()
print(f"  Train: {X_train.shape[0]} | Test: {X_test.shape[0]} | Fraud rate: {prior_fraud:.3f}")

# ── 2. Train classifiers ──
print("[2/5] Training 6 classifiers...")
models = get_all_models(preprocessor)
fitted_models = {}
model_predictions = {}
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    fitted_models[name] = pipeline
    y_pred = pipeline.predict(X_test)
    model_predictions[name] = y_pred
    f1 = f1_score(y_test, y_pred, zero_division=0)
    print(f"  {name}: F1={f1:.4f}")

# ── 3. Bayesian analysis ──
print("[3/5] Bayesian conditionals & posterior updates...")
conditionals = compute_class_conditionals(X_train, y_train, BAYESIAN_FEATURES)
update_traces = compute_posterior_updates(prior_fraud, conditionals, X_test.iloc[:3], BAYESIAN_FEATURES)

# ── 4. Statistical comparison ──
print("[4/5] McNemar tests + cross-validation...")
model_names_list = list(fitted_models.keys())
mcnemar_df = run_mcnemar_matrix(model_predictions, y_test.values, model_names_list)
cv_results = run_cross_validation(models, X, y)

# ── 5. Causal analysis ──
print("[5/5] Causal DAG + propensity scores + feature importance...")
dag = build_causal_dag()
ps_results = propensity_score_analysis(df_full, TREATMENT_VAR, "fraud_reported", CONFOUNDERS)
lr_pipeline = fitted_models["Logistic Regression"]
importance_df = compute_feature_importance_comparison(lr_pipeline, X_test, y_test, conditionals, BAYESIAN_FEATURES)

print("\nSetup complete. Run any figure cell below.")

---
### Figure 1 — Class-Conditional Densities
KDE estimates of P(feature | fraud) vs P(feature | not fraud). Features where the two curves diverge carry discriminative power.

In [ ]:
valid = [f for f in BAYESIAN_FEATURES if f in conditionals]
rows, cols = (len(valid) + 1) // 2, 2
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = np.array(axes).flatten()

for i, feat in enumerate(valid):
    ax = axes[i]
    c = conditionals[feat]
    ax.fill_between(c["x_range"], c["fraud_density"], alpha=0.4, color="#d62728", label="P(x | Fraud)")
    ax.fill_between(c["x_range"], c["legit_density"], alpha=0.4, color="#1f77b4", label="P(x | Not Fraud)")
    ax.plot(c["x_range"], c["fraud_density"], color="#d62728", linewidth=1.5)
    ax.plot(c["x_range"], c["legit_density"], color="#1f77b4", linewidth=1.5)
    ax.set_title(feat.replace("_", " ").title(), fontweight="bold")
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(alpha=0.2, linestyle="--")
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Class-Conditional Distributions", fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "class_conditional_densities")
plt.show()

### Figure 2 — Prior-to-Posterior Bayesian Update
How P(fraud | x) evolves as each feature value is observed, starting from the base rate prior.

In [ ]:
fig, axes = plt.subplots(1, len(update_traces), figsize=(6 * len(update_traces), 5), squeeze=False)
axes = axes.flatten()

for idx, trace in enumerate(update_traces):
    ax = axes[idx]
    labels = [t["feature"].replace("_", " ").title() for t in trace]
    probs = [t["probability"] for t in trace]
    colors = ["#d62728" if p > 0.5 else "#1f77b4" for p in probs]
    bars = ax.bar(range(len(probs)), probs, color=colors, alpha=0.8, edgecolor="black", linewidth=0.5)
    ax.axhline(y=0.5, color="black", linestyle="--", linewidth=1, alpha=0.6)
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=10)
    ax.set_ylabel("P(Fraud)")
    ax.set_ylim(0, 1.05)
    ax.set_title(f"Sample {idx + 1}", fontweight="bold")
    ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
    ax.grid(axis="y", alpha=0.2, linestyle="--")
    for bar, prob in zip(bars, probs):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f"{prob:.3f}", ha="center", fontsize=9)

fig.suptitle("Bayesian Posterior Update — Sequential Feature Evidence", fontsize=15, fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "prior_posterior_update")
plt.show()

---
### Figure 3 — ROC Curves
Receiver Operating Characteristic curves for all classifiers. Higher AUC = better discrimination.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for (name, model), color in zip(fitted_models.items(), ADV_COLORS):
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_proba)
        ax.plot(fpr, tpr, color=color, linewidth=2, label=f"{name} (AUC={auc(fpr,tpr):.3f})")
    except Exception:
        continue
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Random")
ax.set_xlabel("False Positive Rate"); ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison", fontweight="bold")
ax.legend(loc="lower right", fontsize=10)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(alpha=0.2, linestyle="--")
fig.tight_layout()
save_fig(fig, "roc_overlay")
plt.show()

### Figure 4 — Calibration Curves
Predicted probability vs actual fraud frequency. Points on the diagonal = well-calibrated.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for (name, model), color in zip(fitted_models.items(), ADV_COLORS):
    try:
        y_proba = model.predict_proba(X_test)[:, 1]
        if len(np.unique(y_proba)) < 3:
            continue
        prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=8)
        ax.plot(prob_pred, prob_true, "o-", color=color, linewidth=2, label=name)
    except Exception:
        continue
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Perfect")
ax.set_xlabel("Mean Predicted Probability"); ax.set_ylabel("Fraction of Positives")
ax.set_title("Calibration Curves", fontweight="bold")
ax.legend(loc="upper left", fontsize=10)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(alpha=0.2, linestyle="--")
fig.tight_layout()
save_fig(fig, "calibration_curves")
plt.show()

### Figure 5 — Cross-Validation F1 Boxplot
Distribution of F1 scores across 5 stratified folds.

In [ ]:
model_names_cv = list(cv_results.keys())
f1_data = [cv_results[n]["f1_scores"] for n in model_names_cv]

fig, ax = plt.subplots(figsize=(10, 6))
bp = ax.boxplot(f1_data, labels=model_names_cv, patch_artist=True, widths=0.6)
for patch, color in zip(bp["boxes"], ADV_COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.6)
for i, name in enumerate(model_names_cv):
    m, s = cv_results[name]["f1_mean"], cv_results[name]["f1_std"]
    ax.text(i+1, m + s + 0.02, f"{m:.3f}\n(+/-{s:.3f})", ha="center", fontsize=9, fontweight="bold")
ax.set_ylabel("F1 Score")
ax.set_title("5-Fold Cross-Validation F1 Scores", fontweight="bold")
ax.tick_params(axis="x", rotation=15)
ax.spines["top"].set_visible(False); ax.spines["right"].set_visible(False)
ax.grid(axis="y", alpha=0.2, linestyle="--")
fig.tight_layout()
save_fig(fig, "cv_boxplot")
plt.show()

### Figure 6 — McNemar's Test Heatmap
Pairwise p-values testing whether two classifiers make significantly different errors. * = p < 0.05.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
data = mcnemar_df.values.copy()
mask = np.isnan(data)
display_data = np.where(mask, 0.5, data)
im = ax.imshow(display_data, cmap="RdYlGn", vmin=0, vmax=1, aspect="auto")
n = len(mcnemar_df)
for i in range(n):
    for j in range(n):
        if i == j:
            ax.text(j, i, "---", ha="center", va="center", fontsize=10, color="gray")
        else:
            val = data[i, j]
            ax.text(j, i, f"{val:.3f}{' *' if val < 0.05 else ''}", ha="center", va="center",
                    fontsize=9, color="white" if val < 0.05 else "black",
                    fontweight="bold" if val < 0.05 else "normal")
names = list(mcnemar_df.columns)
ax.set_xticks(range(n)); ax.set_yticks(range(n))
ax.set_xticklabels(names, rotation=30, ha="right"); ax.set_yticklabels(names)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).set_label("p-value")
ax.set_title("McNemar's Test p-values (* = p<0.05)", fontweight="bold")
fig.tight_layout()
save_fig(fig, "mcnemar_heatmap")
plt.show()

---
### Figure 7 — Causal DAG
Hypothesised causal structure for insurance fraud, based on domain knowledge.

In [ ]:
layers = {
    0: ["insured_occupation"],
    1: ["policy_annual_premium", "policy_deductable", "policy_age_days"],
    2: ["umbrella_limit", "incident_severity", "number_of_vehicles"],
    3: ["total_claim_amount", "bodily_injuries", "witnesses", "police_report_available"],
    4: ["fraud_reported"],
}
pos = {}
for layer_idx, nodes in layers.items():
    y_pos = 1.0 - layer_idx * 0.22
    for i, node in enumerate(nodes):
        pos[node] = ((i - (len(nodes)-1)/2) * 0.28, y_pos)

fig, ax = plt.subplots(figsize=(14, 10))
node_colors = ["#d62728" if n == "fraud_reported" else "#6baed6" for n in dag.nodes()]
node_sizes = [3000 if n == "fraud_reported" else 2000 for n in dag.nodes()]
nx.draw_networkx_nodes(dag, pos, ax=ax, node_color=node_colors, node_size=node_sizes, alpha=0.9, edgecolors="black", linewidths=1.2)
nx.draw_networkx_edges(dag, pos, ax=ax, edge_color="#555", arrows=True, arrowsize=20, width=1.5,
                       connectionstyle="arc3,rad=0.1", min_source_margin=20, min_target_margin=20)
labels = {n: n.replace("_", "\n") for n in dag.nodes()}
nx.draw_networkx_labels(dag, pos, labels=labels, ax=ax, font_size=10, font_weight="bold")
ax.set_title("Hypothesised Causal DAG for Insurance Fraud", fontweight="bold", pad=20)
ax.axis("off")
fig.tight_layout()
save_fig(fig, "causal_dag")
plt.show()

### Figure 8 — Propensity Score Analysis
Left: P(police report | covariates) distributions. Right: stratum-level treatment effects on fraud.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ps = ps_results["propensity_scores"]
treatment = ps_results["treatment"]

ax1.hist(ps[treatment == 1], bins=20, alpha=0.6, color="#1f77b4", label="Report=YES", density=True)
ax1.hist(ps[treatment == 0], bins=20, alpha=0.6, color="#d62728", label="Report=NO", density=True)
ax1.set_xlabel("Propensity Score"); ax1.set_ylabel("Density")
ax1.set_title("Propensity Score Distributions", fontweight="bold")
ax1.legend(); ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
ax1.grid(alpha=0.2, linestyle="--")

effects = ps_results["stratum_effects"]
ate = ps_results["ate"]
colors_ps = ["#2ca02c" if e > 0 else "#d62728" for e in effects]
ax2.bar(range(len(effects)), effects, color=colors_ps, alpha=0.7, edgecolor="black", linewidth=0.5)
ax2.axhline(y=ate, color="black", linestyle="--", linewidth=2, label=f"ATE = {ate:.4f}")
ax2.axhline(y=0, color="gray", linestyle="-", linewidth=0.5)
ax2.set_xlabel("Propensity Score Stratum (Quintile)"); ax2.set_ylabel("Treatment Effect")
ax2.set_title("Stratum-Level Treatment Effects", fontweight="bold")
ax2.set_xticks(range(len(effects))); ax2.set_xticklabels([f"Q{i+1}" for i in range(len(effects))])
ax2.legend(); ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
ax2.grid(axis="y", alpha=0.2, linestyle="--")

fig.suptitle("Propensity Score Analysis: Effect of Police Report on Fraud", fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "propensity_scores")
plt.show()

### Figure 9 — Feature Importance Comparison
Left: permutation importance (how much F1 drops). Right: KL divergence between class-conditional distributions.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

top_perm = importance_df.dropna(subset=["perm_importance"]).nlargest(12, "perm_importance")
ax1.barh(range(len(top_perm)), top_perm["perm_importance"].values, color="#1f77b4", alpha=0.7)
ax1.set_yticks(range(len(top_perm)))
ax1.set_yticklabels([f.replace("_", " ").title() for f in top_perm["feature"]], fontsize=10)
ax1.invert_yaxis()
ax1.set_xlabel("Permutation Importance (F1 drop)")
ax1.set_title("Permutation Importance\n(Logistic Regression)", fontweight="bold")
ax1.spines["top"].set_visible(False); ax1.spines["right"].set_visible(False)
ax1.grid(axis="x", alpha=0.2, linestyle="--")

kl_data = importance_df.dropna(subset=["kl_divergence"]).sort_values("kl_divergence", ascending=False)
if len(kl_data) > 0:
    ax2.barh(range(len(kl_data)), kl_data["kl_divergence"].values, color="#d62728", alpha=0.7)
    ax2.set_yticks(range(len(kl_data)))
    ax2.set_yticklabels([f.replace("_", " ").title() for f in kl_data["feature"]], fontsize=10)
    ax2.invert_yaxis()
ax2.set_xlabel("KL Divergence")
ax2.set_title("Bayesian Likelihood Divergence\n(Class-Conditional KDE)", fontweight="bold")
ax2.spines["top"].set_visible(False); ax2.spines["right"].set_visible(False)
ax2.grid(axis="x", alpha=0.2, linestyle="--")

fig.suptitle("Feature Importance: Discriminative vs. Bayesian", fontweight="bold", y=1.02)
fig.tight_layout()
save_fig(fig, "feature_importance_comparison")
plt.show()